# Kestrel distill collection, option c: a teacher on Kaggle (no Groq)

Two modes (known issue #72, `distill/KAGGLE.md`):
- **`MODE = "compare"`**: #72's pre-registered teacher selection. Both candidates (gpt-oss:20b and
  Qwen3-30B-A3B-Instruct-2507 at Q4) run the same 20 check-graded distill tasks once, `--judge none`; the
  last cell prints the table and the winner by #72's rule. Never KestrelBench.
- **`MODE = "collect"`**: the full distill pass (122 tasks) with `TEACHER`, in shards, so a session that ends
  early loses at most one shard; attach this notebook's previous output and run it again to resume.

Settings: **Accelerator: GPU T4 x2**, **Internet: on** (uv, Python, Ollama, the model pull and the 7 web
tasks; the switch needs a verified phone number). No keys of any kind: the notebook sets none, so no hosted
model can be called. Run it with **Save Version > Save & Run All (Commit)**: only a saved version keeps its
outputs after the session ends.

Input: the **kestrel-snapshot** dataset (`kestrel-snapshot.zip` from `git archive`, or its extracted tree)
with its `COMMIT` file; for a resume, also this notebook's previous output.

In [ ]:
# Settings. Only MODE and TEACHER normally change.
MODE = "compare"  # "compare" (#72's teacher selection) or "collect" (the full pass with TEACHER)
# collect mode: compare mode's winner, as #72 names it ("gpt-oss:20b" or "Qwen3-30B-A3B-Instruct-2507")
TEACHER = "gpt-oss:20b"
SHARDS = 8  # collect mode: the 122 tasks in this many parts (--shard i/SHARDS)

OLLAMA_VERSION = "0.40.13"  # the laptop's; the notebook stops on any other
UV_VERSION = "0.12.22"
PYTHON_VERSION = "3.14"  # Kestrel's .python-version
NUM_CTX = 16384  # the context Kestrel's history budget (16k) needs
REQUEST_TIMEOUT_S = 600  # per model request; a T4 is slow on the first long prompt

assert MODE in ("compare", "collect"), MODE

In [ ]:
# Paths, helpers, and the code snapshot. Code and models go to /tmp (big, and not part of the outputs);
# only /kaggle/working/out is saved with the version.
import datetime
import json
import os
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
REPO = Path("/tmp/kestrel")
MODELS = Path("/tmp/ollama-models")
OUT = Path("/kaggle/working/out")
OUT.mkdir(parents=True, exist_ok=True)
STARTED = datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds")


def sh(*args, cwd=None, env=None, quiet=False):
    # Runs a command, streaming its output (or only its last lines if quiet); stops the notebook if it fails.
    lines = []
    with subprocess.Popen(
        [str(a) for a in args], cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as p:
        for line in p.stdout:
            lines.append(line)
            if not quiet:
                print(line, end="")
    if quiet:
        print("".join(lines[-5:]), end="")
    if p.returncode:
        raise RuntimeError(
            f"{Path(str(args[0])).name} {args[1] if len(args) > 1 else ''} exited with {p.returncode}: see above"
        )
    return "".join(lines)


def find_snapshot():
    trees = sorted(
        {
            Path(d)
            for d, _, f in os.walk(INPUT, followlinks=True)
            if "pyproject.toml" in f and Path(d, "src", "kestrel", "__init__.py").is_file()
        }
    )
    zips = sorted(p for p in INPUT.rglob("kestrel-snapshot*.zip"))
    if len(trees) + len(zips) != 1:
        found = [str(p) for p in trees + zips] or "none"
        raise RuntimeError(
            f"expected one Kestrel snapshot under {INPUT} (a git-archive zip or its tree); found {found}"
        )
    shutil.rmtree(REPO, ignore_errors=True)
    if zips:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(REPO)
        return zips[0]
    shutil.copytree(trees[0], REPO)
    return trees[0]


source = find_snapshot()
commit_file = REPO / "COMMIT"
if not commit_file.is_file():
    raise RuntimeError(
        "the snapshot has no COMMIT file: make it with `git archive --add-virtual-file=COMMIT:<sha>` "
        "(distill/KAGGLE.md, step 1)"
    )
COMMIT = commit_file.read_text().strip()
print(f"snapshot: {source}\ncommit:   {COMMIT}\ncode in:  {REPO}\noutputs:  {OUT}")

# A resume: files this notebook saved in an earlier version (attached as input) are copied back first.
previous = [
    p
    for p in INPUT.rglob("*")
    if p.is_file() and "out" in p.parts and REPO not in p.parents and (p.suffix in (".json", ".db", ".md"))
]
for p in previous:
    rel = Path(*p.parts[p.parts.index("out") + 1 :])
    target = OUT / rel
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, target)
print(
    f"resumed files from earlier output: {len(previous)}" if previous else "no earlier output attached (a fresh start)"
)

In [ ]:
# GPU and internet. The free tier's GPU T4 x2 is two 16 GB cards.
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
)
GPUS = (
    [{"name": n.strip(), "memory": m.strip()} for n, m in (line.split(",") for line in smi.stdout.strip().splitlines())]
    if smi.returncode == 0
    else []
)
print(GPUS or "no GPU")
if len(GPUS) < 2 or not all("T4" in g["name"] for g in GPUS):
    raise RuntimeError("set Accelerator to GPU T4 x2 (#72's fit rule is about Kaggle's two free T4s)")
for host in ("pypi.org", "github.com", "ollama.com", "registry.ollama.ai", "duckduckgo.com"):
    try:
        socket.getaddrinfo(host, 443)
    except OSError as e:
        raise RuntimeError(
            f"No internet (can't resolve {host}: {e}). Turn on Settings > Internet (needs a verified phone "
            "number), then run again from the top."
        ) from None
print("internet: ok")

In [ ]:
# Kestrel in its own Python 3.14 (Kaggle's kernel Python is older): uv at its pin, exactly what uv.lock says.
sh(sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_VERSION}")
UV = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
found = sh(UV, "--version", quiet=True).split()[1]
if found != UV_VERSION:
    raise RuntimeError(f"uv {found}, pin {UV_VERSION}")
sh(UV, "python", "install", PYTHON_VERSION, quiet=True)
sh(UV, "sync", "--locked", "--no-dev", "--python", PYTHON_VERSION, cwd=REPO, quiet=True)
PY = REPO / ".venv" / "bin" / "python"
KESTREL = REPO / ".venv" / "bin" / "kestrel"


def venv(code):
    # Runs Python code in Kestrel's venv (from the repo root) and returns its JSON output.
    return json.loads(sh(PY, "-c", code, cwd=REPO, quiet=True).strip().splitlines()[-1])


INFO = venv(
    "import json, sys, dataclasses; sys.path.insert(0, 'distill'); import kaggle_collect as k; "
    "print(json.dumps({'slice': list(k.SLICE), 'candidates': [dataclasses.asdict(c) for c in k.CANDIDATES], "
    "'python': sys.version.split()[0]}))"
)
print(f"Kestrel {COMMIT[:12]} on Python {INFO['python']}; candidates: {[c['name'] for c in INFO['candidates']]}")

In [ ]:
# Ollama at its pin, serving on localhost only. Models go to /tmp (a 30B model doesn't fit /kaggle/working's quota).
# Ollama's install script needs zstd to unpack, and Kaggle's image doesn't have it.
if not shutil.which("zstd"):
    sh("bash", "-c", "apt-get update -qq && apt-get install -y -qq zstd", quiet=True)
sh("bash", "-c", f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh", quiet=True)
version = sh("ollama", "--version", quiet=True)
if OLLAMA_VERSION not in version:
    raise RuntimeError(f"Ollama {version.strip()}, pin {OLLAMA_VERSION}")
MODELS.mkdir(parents=True, exist_ok=True)
OLLAMA_ENV = os.environ | {
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": str(MODELS),
    "OLLAMA_CONTEXT_LENGTH": str(NUM_CTX),
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_NUM_PARALLEL": "1",
}
log = open(OUT / "ollama.log", "a")  # noqa: SIM115 (kept open for the server's whole life)
server = subprocess.Popen(["ollama", "serve"], env=OLLAMA_ENV, stdout=log, stderr=subprocess.STDOUT)
API = "http://127.0.0.1:11434"


def api(path, body=None):
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(API + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT_S) as r:
        return json.loads(r.read() or b"{}")


for _ in range(60):
    try:
        print("Ollama", api("/api/version")["version"])
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve didn't start: see out/ollama.log")

In [ ]:
# Pull a candidate, serve it the way kaggle_collect.CANDIDATES says, and check it fits the two T4s.
def prepare(cand):
    sh("ollama", "pull", cand["tag"], env=OLLAMA_ENV, quiet=True)
    digest = next((m.get("digest") for m in api("/api/tags")["models"] if m["name"] == cand["tag"]), None)
    if cand["template"] == "kestrel":  # known issue #73: the template the student is trained on
        tag = cand["tag"]
        modelfile = venv(
            "import json, sys; sys.path.insert(0, 'distill'); from export import modelfile; "
            "from kestrel.distill import train_format as f; "
            f"print(json.dumps(modelfile(f.OLLAMA_TEMPLATE, 'x', {NUM_CTX}).replace('FROM ./x', 'FROM ' + {tag!r})))"
        )
        # export.modelfile's LICENSE line describes the student; a teacher keeps its package's license
        modelfile = "".join(line for line in modelfile.splitlines(keepends=True) if not line.startswith("LICENSE"))
        path = Path("/tmp") / f"{cand['served_as']}.Modelfile"
        path.write_text(modelfile)
        sh("ollama", "create", cand["served_as"], "-f", path, env=OLLAMA_ENV, quiet=True)
    t0 = time.time()
    api(
        "/api/chat", {"model": cand["served_as"], "messages": [{"role": "user", "content": "Say ok."}], "stream": False}
    )
    loaded = next(
        (m for m in api("/api/ps")["models"] if m["name"].split(":")[0] == cand["served_as"].split(":")[0]), {}
    )
    size, vram = int(loaded.get("size") or 0), int(loaded.get("size_vram") or 0)
    fit = size > 0 and vram >= size  # kaggle_collect.fits(): nothing spilled to the CPU
    print(
        f"{cand['name']}: {size / 2**30:.1f} GiB loaded, {vram / 2**30:.1f} GiB on the GPUs -> "
        f"{'fits' if fit else 'DOES NOT FIT the two T4s'} (first answer {time.time() - t0:.0f} s)"
    )
    return digest, fit, {"size": size, "size_vram": vram}


def unload(cand):
    for name in {cand["served_as"], cand["tag"]}:
        subprocess.run(["ollama", "rm", name], env=OLLAMA_ENV, capture_output=True)


BENCH_ENV = {k: v for k, v in os.environ.items() if not any(s in k.upper() for s in ("KEY", "TOKEN", "SECRET"))} | {
    "KESTREL_MCP": "off",
    "OLLAMA_BASE_URL": f"{API}/v1",
    "KESTREL_REQUEST_TIMEOUT": str(REQUEST_TIMEOUT_S),
}


def bench(cwd, *args):
    cwd.mkdir(parents=True, exist_ok=True)  # logs/distill.db lands under cwd
    sh(
        KESTREL,
        "bench",
        "run",
        "--split",
        "distill",
        "--provider",
        "ollama",
        "--judge",
        "none",
        "--stop-after-errors",
        "0",
        *args,
        cwd=cwd,
        env=BENCH_ENV,
    )


def complete(path):
    if not path.is_file():
        return False
    rows = json.loads(path.read_text()).get("tasks", [])
    return bool(rows) and all(r["status"] in ("pass", "fail", "excluded") for r in rows)


def write_manifest(folder, cand, digest, fit, loaded, files):
    agent = None
    for f in files:
        agent = agent or (json.loads((folder / f).read_text()).get("meta", {}).get("agent") or {}).get("sha")
    manifest = {
        "mode": MODE,
        "teacher": cand | {"digest": digest, "provider": "ollama"},
        "ollama_version": OLLAMA_VERSION,
        "gpu": {"devices": GPUS, "fits": fit, "loaded": loaded},
        "commit": COMMIT,
        "agent_fingerprint": agent,
        "files": files,
        "started": STARTED,
        "finished": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    }
    (folder / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
    return manifest

## Compare mode: #72's teacher selection

Each candidate: pull, serve, fit check, then the 20 slice tasks once. A candidate that doesn't fit the two T4s
doesn't run (it is out by #72's rule). Then `kaggle_collect.py select` applies the rule. About 1-2 h in total.

In [ ]:
if MODE != "compare":
    print("compare mode skipped (MODE is collect)")
else:
    folders = []
    for cand in INFO["candidates"]:
        folder = OUT / "compare" / cand["served_as"].replace(":", "-")
        folder.mkdir(parents=True, exist_ok=True)
        folders.append(folder)
        if (folder / "manifest.json").is_file():
            done = json.loads((folder / "manifest.json").read_text())
            if complete(folder / "results.json") or done["gpu"]["fits"] is False:
                print(f"{cand['name']}: already measured (resumed), skipped")
                continue
        digest, fit, loaded = prepare(cand)
        files = []
        if fit:
            bench(
                folder,
                "--tasks",
                ",".join(INFO["slice"]),
                "--model",
                cand["served_as"],
                "--out",
                folder / "results.json",
            )
            files = ["results.json"]
        write_manifest(folder, cand, digest, fit, loaded, files)
        unload(cand)  # frees /tmp before the next pull
    sh(
        PY,
        "distill/kaggle_collect.py",
        "select",
        *[a for f in folders for a in ("--candidate", f)],
        "--out",
        OUT / "compare",
        cwd=REPO,
    )

## Collect mode: the full pass with `TEACHER`

Runs only if `teachers.toml` in the snapshot lists exactly that teacher (otherwise every trace would be dropped
by `kestrel distill build`). Shards already complete in an attached earlier output are skipped; a partial one
is resumed with `--resume`.

In [ ]:
if MODE != "collect":
    print("collect mode skipped (MODE is compare)")
else:
    cand = next((c for c in INFO["candidates"] if TEACHER in (c["name"], c["tag"], c["served_as"])), None)
    if cand is None:
        raise RuntimeError(f"TEACHER {TEACHER!r} is not a #72 candidate")
    teachers = venv(
        "import json; from kestrel.distill.dataset import load_teachers; print(json.dumps(sorted(load_teachers())))"
    )
    if teachers != [["ollama", cand["served_as"]]]:
        raise RuntimeError(
            f"teachers.toml in this snapshot lists {teachers}, not [['ollama', {cand['served_as']!r}]]: uncomment "
            "the winner's entry (and comment out option a) on the laptop, commit, and upload a new snapshot "
            "(KAGGLE.md, step 6)"
        )
    folder = OUT / "collect"
    digest, fit, loaded = prepare(cand)
    if not fit:
        raise RuntimeError(f"{cand['name']} doesn't fit the two T4s: it can't be the teacher (#72)")
    names = []
    for i in range(1, SHARDS + 1):
        served = cand["served_as"]
        name = venv(
            "import json, sys; sys.path.insert(0, 'distill'); import kaggle_collect as k; "
            f"print(json.dumps(k.shard_file({served!r}, {i}, {SHARDS})))"
        )
        names.append(f"results/{name}")
        path = folder / "results" / name
        if complete(path):
            print(f"shard {i}/{SHARDS}: complete (resumed), skipped")
            continue
        t0 = time.time()
        common = ["--model", cand["served_as"], "--shard", f"{i}/{SHARDS}"]
        bench(folder, *common, *(["--resume", path] if path.is_file() else ["--out", path]))
        print(f"shard {i}/{SHARDS}: {(time.time() - t0) / 60:.0f} min")
        write_manifest(folder, cand, digest, fit, loaded, [n for n in names if (folder / n).is_file()])
    write_manifest(folder, cand, digest, fit, loaded, names)
    sh(PY, "distill/kaggle_collect.py", "check-shards", *[folder / n for n in names], cwd=REPO)

In [ ]:
# What to download (Output tab, or the version's Output page): one zip of out/<mode>.
server.terminate()
folder = OUT / MODE
if not folder.exists():
    raise RuntimeError(f"nothing in {folder}")
archive = shutil.make_archive(f"/kaggle/working/kestrel-{MODE}-{COMMIT[:12]}", "zip", OUT, MODE)
print(f"Download: {archive}")
for p in sorted(folder.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  {p.stat().st_size / 2**20:.1f} MiB")
print("Where each file goes on the laptop: distill/KAGGLE.md, step 5 (compare) or step 8 (collect).")